In [ ]:
!pip install ultralytics --quiet
!pip install scipy --quiet
!pip install opencv-python-headless --quiet
!pip install matplotlib tqdm --quiet

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1: IMPORTS & GLOBAL CONFIG
# Paper: "Structured Universal Adversarial Attacks" (arXiv:2510.14460v1)
# ═══════════════════════════════════════════════════════════════════════════
import os, sys, json, shutil, random, time, re
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from tqdm import tqdm
from ultralytics import YOLO

# ── Thư viện Excel ────────────────────────────────────────────────────────
try:
    import openpyxl
    from openpyxl.styles import (PatternFill, Font, Alignment,
                                  Border, Side, numbers)
    from openpyxl.chart import BarChart, LineChart, Reference
    from openpyxl.utils import get_column_letter
    XLSX_OK = True
except ImportError:
    XLSX_OK = False
    print("[WARN] openpyxl not found — Excel export disabled. "
          "pip install openpyxl")

# ══════════════════════════════════════════════════════════════════════════
# USER CONFIG 
# ══════════════════════════════════════════════════════════════════════════
DATASET_ROOT  = "/kaggle/input/datasets/bichhanhluuthi/my-dataset/My_Thesis_Dataset"
PERTURBATION_DIR = Path("/kaggle/input/datasets/bichhanhluuthi/best-uap")
OUTPUT_DIR    = Path("/kaggle/working/EVAL-AO-Exp-Attack")
YOLO_MODEL    = "yolo11m.pt"
IMG_SIZE      = 640
CONF_THRESHOLD = 0.25   
IOU_THRESHOLD  = 0.5
SEED          = 42      
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
# TEST_RATIO  = 0.15    # implied
MAX_VIZ_FRAMES = 20     
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VIDEO_NAMES = [
    "annichkov_most_001",
    "annichkov_most_002",
    "dvorzovy_most_001",
    "gostiny_dvor_001",
    "gostiny_dvor_002",
    "zagorodny_proezd_001",
]


VIDEO_CLASS_MAP: Dict[str, List[str]] = {
    "annichkov_most_001" : ["person", "vehicle"],
    "annichkov_most_002" : ["person", "vehicle"],
    "dvorzovy_most_001"  : ["vehicle"],           # ← only vehicle
    "gostiny_dvor_001"   : ["person", "vehicle"],
    "gostiny_dvor_002"   : ["person", "vehicle"],
    "zagorodny_proezd_001": ["person", "vehicle"],
}

YOLO_VEHICLE_IDS = {2, 5, 7}   # car, bus, truck
YOLO_PERSON_IDS  = {0}
CLASS_MODE_TO_YOLO = {
    "person" : YOLO_PERSON_IDS,
    "vehicle": YOLO_VEHICLE_IDS,
}
JSON_TO_YOLO_LABEL = {2: 0, 1: 1}   # person→0, vehicle→1
YOLO_LABEL_NAMES   = ["person", "vehicle"]

VIDEO_FRAME_COUNTS = {
    "annichkov_most_001" : 914,
    "annichkov_most_002" : 3607,
    "dvorzovy_most_001"  : 1158,
    "gostiny_dvor_001"   : 1926,
    "gostiny_dvor_002"   : 2461,
    "zagorodny_proezd_001": 912,
}

# ── Seed ──────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 62)
print("  UAP Evaluation Pipeline — Independent Test Set")
print("  Paper: Structured Universal Adversarial Attacks")
print("  arXiv:2510.14460v1 | Jacob et al. (2025)")
print("=" * 62)
print(f"  Device   : {DEVICE}")
print(f"  YOLO     : {YOLO_MODEL}")
print(f"  τ (conf) : {CONF_THRESHOLD}")
print(f"  Dataset  : {DATASET_ROOT}")
print(f"  UAP dir  : {PERTURBATION_DIR}")
print(f"  Output   : {OUTPUT_DIR}")
print(f"  Seed     : {SEED}  (must match training seed)")
print("=" * 62)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — DATASET SPLIT 70/15/15 
# ═══════════════════════════════════════════════════════════════════════════

def coco_bbox_to_yolo(bbox, img_w, img_h):
    """
    Convert absolute [x, y, w, h] bounding box to normalized [cx, cy, w, h] format.
    Copied verbatim from Cell 5 — must remain unmodified.
    """
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    w  =  w / img_w
    h  =  h / img_h
    cx = max(0.0, min(1.0, cx))
    cy = max(0.0, min(1.0, cy))
    w  = max(0.0, min(1.0,  w))
    h  = max(0.0, min(1.0,  h))
    return cx, cy, w, h


def build_yolo_dataset_and_get_test(
        dataset_root: str,
        output_dir: Path,
        train_r: float = TRAIN_RATIO,
        val_r:   float = VAL_RATIO,
) -> Tuple[Path, Dict[str, List[str]]]:
    """
    Faithfully reproduce the data splitting logic from Cell 5 of the original notebook:
      - Iterate through the 6 videos in the order of VIDEO_NAMES.
      - Apply random.shuffle(paired) using the same random seed.
      - Perform a 70/15/15 data split.
      - Write train.txt, val.txt, and test.txt.
      - Return (dataset_dir, {video_name: [test_img_path, ...]}).
      
    Note: This function does not read the train/val files; it only returns the test image paths.
    """
    dataset_root = Path(dataset_root)
    output_dir.mkdir(parents=True, exist_ok=True)

    splits = {"train": [], "val": [], "test": []}
    video_test_files: Dict[str, List[str]] = {}   # video → test img paths

    for video_name in VIDEO_NAMES:
        # ── Adaptively detect directory structure (identical to the original) ──
        vid_dir  = dataset_root / video_name
        ann_path = vid_dir / "annotations.json"
        img_dir  = vid_dir / "images"

        if not ann_path.exists():
            alt = dataset_root / "annotations" / f"{video_name}.json"
            if alt.exists(): ann_path = alt
        if not img_dir.exists():
            alt = dataset_root / "images" / video_name
            if alt.exists(): img_dir = alt

        if not ann_path.exists() or not img_dir.exists():
            print(f"  [WARN] Missing data for {video_name} → skipping")
            continue

        with open(ann_path, "r") as f:
            coco_data = json.load(f)

        img_idx = {img["id"]: (img["width"], img["height"], img["file_name"])
                   for img in coco_data.get("images", [])}
        ann_idx = {iid: [] for iid in img_idx}
        for ann in coco_data.get("annotations", []):
            if ann["image_id"] in ann_idx:
                ann_idx[ann["image_id"]].append(ann)

        v_img_dir = output_dir / video_name / "images"
        v_lbl_dir = output_dir / video_name / "labels"
        v_img_dir.mkdir(parents=True, exist_ok=True)
        v_lbl_dir.mkdir(parents=True, exist_ok=True)

        paired = []
        for iid, (iw, ih, fname) in img_idx.items():
            stem = Path(fname).stem
            found_img = None
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                p = img_dir / f"{stem}{ext}"
                if p.exists():
                    found_img = p
                    break
            if not found_img:
                continue

            out_img = v_img_dir / f"{stem}.jpg"
            if not out_img.exists():
                shutil.copy2(found_img, out_img)

            out_lbl = v_lbl_dir / f"{stem}.txt"
            lines   = []
            for ann in ann_idx.get(iid, []):
                cid = ann.get("category_id")
                if cid not in JSON_TO_YOLO_LABEL:
                    continue
                yolo_cls = JSON_TO_YOLO_LABEL[cid]
                cx, cy, bw, bh = coco_bbox_to_yolo(ann["bbox"], iw, ih)
                lines.append(f"{yolo_cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            out_lbl.write_text("\n".join(lines))
            paired.append((str(out_img), str(out_lbl)))

        # ── Perform 70/15/15 split using the same seed (in-place shuffle) ──
        random.shuffle(paired)          # ← The random seed must match exactly
        n      = len(paired)
        n_train = int(n * train_r)
        n_val   = int(n * val_r)
        train_p = paired[:n_train]
        val_p   = paired[n_train : n_train + n_val]
        test_p  = paired[n_train + n_val :]

        splits["train"].extend(train_p)
        splits["val"].extend(val_p)
        splits["test"].extend(test_p)

        test_img_paths = [p[0] for p in test_p]
        video_test_files[video_name] = sorted(test_img_paths,
                                               key=lambda p: Path(p).name)

        print(f"  [{video_name}] train={len(train_p)} | "
              f"val={len(val_p)} | test={len(test_p)}")

    # Write the split text files (for reference and debugging)
    for sname, pairs in splits.items():
        (output_dir / f"{sname}.txt").write_text(
            "\n".join(p[0] for p in pairs))

    total = {k: len(v) for k, v in splits.items()}
    print(f"\n[DATA] Done → train={total['train']} | "
          f"val={total['val']} | test={total['test']}")
    print(f"[DATA] Test leakage check: "
          f"train∩test = 0 (guaranteed by sequential split)")

    return output_dir, video_test_files


# ── Dedicated DATASET_DIR for evaluation (isolated from the training set) ──
EVAL_DATASET_DIR = OUTPUT_DIR / "dataset_eval_split"

print("\n[STEP 1] Reproducing 70/15/15 split (seed-exact) ...")
eval_dataset_dir, video_test_files = build_yolo_dataset_and_get_test(
    DATASET_ROOT, EVAL_DATASET_DIR
)

print(f"\n[INFO] Test set per video:")
for v, paths in video_test_files.items():
    print(f"  {v:<30} {len(paths):>4} frames")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — Scan the perturbations directory, parse filenames, and map UAPs
# Naming convention: uap_{video_name}_{class_mode}.pt
# ═══════════════════════════════════════════════════════════════════════════

def scan_perturbation_dir(perturb_dir: Path) -> Dict[Tuple[str,str], Path]:
    """
    Scan the directory and parse filenames matching the pattern uap_{video}_{class}.pt.
    Returns a dictionary mapping (video_name, class_mode) to the corresponding file Path.
    """
    perturb_dir = Path(perturb_dir)
    if not perturb_dir.exists():
        raise FileNotFoundError(f"Perturbation directory not found: {perturb_dir}")

    found: Dict[Tuple[str,str], Path] = {}
    pattern = re.compile(r"^uap_(.+?)_(person|vehicle)\.pt$")

    for fpath in sorted(perturb_dir.glob("uap_*.pt")):
        m = pattern.match(fpath.name)
        if not m:
            print(f"  [WARN] Filename does not conform to the standard convention, skipping: {fpath.name}")
            continue
        video_name = m.group(1)
        class_mode = m.group(2)

        # Validation: Ensure the video is in the predefined list
        if video_name not in VIDEO_NAMES:
            print(f"  [WARN] Unrecognized video: {video_name}")
            continue
        # Validation: Ensure the class is valid according to VIDEO_CLASS_MAP
        allowed = VIDEO_CLASS_MAP.get(video_name, [])
        if class_mode not in allowed:
            print(f"  [WARN] {video_name} does not contain class='{class_mode}' "
                  f"(allowed: {allowed}) → skipping")
            continue

        found[(video_name, class_mode)] = fpath
        print(f"  [FOUND] {fpath.name:<55} → ({video_name}, {class_mode})")

    return found


def load_uap(fpath: Path, device: torch.device) -> Tuple[torch.Tensor, Dict]:
    """
    Load the .pt file and return a tuple containing the delta tensor on the specified device 
    and a metadata dictionary. 
    The delta tensor has the shape (C, H, W) = (3, 640, 640).
    """
    payload = torch.load(str(fpath), map_location="cpu", weights_only=False)
    delta   = payload["delta"].to(device)          # (3, H, W)
    meta    = {
        "video_name" : payload.get("video_name", "?"),
        "class_mode" : payload.get("class_mode", "?"),
        "metrics"    : payload.get("metrics", {}),
        "hyperparams": payload.get("hyperparams", {}),
        "timestamp"  : payload.get("timestamp", "?"),
        "shape"      : tuple(delta.shape),
    }
    return delta, meta


# ── Validation: Check for missing UAPs ───────────────────────────────────
def check_missing_uaps(found: Dict) -> None:
    expected_pairs = [
        (v, c)
        for v in VIDEO_NAMES
        for c in VIDEO_CLASS_MAP.get(v, [])
    ]
    print(f"\n[UAP SCAN] Total expected: {len(expected_pairs)} | "
          f"Found: {len(found)}")
    missing = [p for p in expected_pairs if p not in found]
    if missing:
        print(f"  ⚠ MISSING {len(missing)} UAPs:")
        for v, c in missing:
            print(f"    - uap_{v}_{c}.pt")
    else:
        print("  ✅ All required UAP files are present.")


print(f"\n[STEP 2] Scanning perturbation directory: {PERTURBATION_DIR}")
uap_map = scan_perturbation_dir(PERTURBATION_DIR)
check_missing_uaps(uap_map)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — YOLO wrapper using only NMS output (not raw anchors)
# ═══════════════════════════════════════════════════════════════════════════

class EvalYOLO:
    """
    Lightweight wrapper for YOLOv11m — utilizes only the high-level ultralytics API.
    Counts bounding boxes based on NMS output (accurate), rather than the raw tensor (8400 anchors).
    """
    def __init__(self,
                 model_name: str       = YOLO_MODEL,
                 img_size: int         = IMG_SIZE,
                 conf_threshold: float = CONF_THRESHOLD,
                 iou_threshold: float  = IOU_THRESHOLD,
                 device: torch.device  = DEVICE):
        self.img_size       = img_size
        self.conf_threshold = conf_threshold
        self.iou_threshold  = iou_threshold
        self.device         = device
        self._yolo = YOLO(model_name)
        print(f"  [EvalYOLO] Loaded {model_name} | "
              f"conf={conf_threshold} | iou={iou_threshold} | "
              f"device={device}")

    def _tensor_to_bgr(self, x: torch.Tensor) -> np.ndarray:
        """Convert a (C, H, W) float tensor in [0, 1] to a BGR uint8 NumPy array."""
        rgb = (x.cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    @torch.no_grad()
    def get_boxes(self,
                  x: torch.Tensor,
                  target_classes: set,
                  conf_thr: float = None) -> List[Dict]:
        """
        Args:
            x: Image tensor of shape (C, H, W) or (1, C, H, W) with values in [0, 1].
            
        Returns:
            A list of dictionaries: {cls, conf, xyxy: [x1, y1, x2, y2]}.
            Only bounding boxes belonging to the target classes after NMS are included.
        """
        if x.dim() == 4:
            x = x.squeeze(0)
        thr  = conf_thr or self.conf_threshold
        img  = self._tensor_to_bgr(x)
        res  = self._yolo(
            img,
            verbose  = False,
            conf     = thr,
            iou      = self.iou_threshold,
            classes  = sorted(target_classes),
            imgsz    = self.img_size,
        )
        boxes = []
        if res and res[0].boxes is not None:
            for b in res[0].boxes:
                boxes.append({
                    "cls" : int(b.cls.item()),
                    "conf": float(b.conf.item()),
                    "xyxy": b.xyxy.squeeze().tolist(),
                })
        return boxes

    @torch.no_grad()
    def count_boxes(self, x: torch.Tensor,
                    target_classes: set,
                    conf_thr: float = None) -> int:
        """Return the number of bounding boxes detected for the target classes after NMS."""
        return len(self.get_boxes(x, target_classes, conf_thr))

    @torch.no_grad()
    def get_annotated_image(self,
                             x: torch.Tensor,
                             boxes: List[Dict],
                             label: str = "",
                             color_bgr: Tuple = (0, 200, 0)) -> np.ndarray:
        """
        Draw bounding boxes on the image and add a header text label.
        Returns a BGR uint8 NumPy array.
        """
        img = self._tensor_to_bgr(x).copy()
        H, W = img.shape[:2]

        for b in boxes:
            x1, y1, x2, y2 = [int(v) for v in b["xyxy"]]
            x1, x2 = max(0,min(W,x1)), max(0,min(W,x2))
            y1, y2 = max(0,min(H,y1)), max(0,min(H,y2))
            cv2.rectangle(img, (x1,y1), (x2,y2), color_bgr, 2)
            lbl_txt = f"cls{b['cls']} {b['conf']:.2f}"
            cv2.putText(img, lbl_txt, (x1, max(y1-4,12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_bgr, 1)

        # Header bar
        cv2.rectangle(img, (0,0), (W, 36), (20,20,20), -1)
        cv2.putText(img, label, (6, 26),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                    color_bgr, 2)
        return img


# Initialize the model once and reuse it for all evaluations
print("\n[STEP 3] Initializing EvalYOLO ...")
eval_yolo = EvalYOLO()
print("  [OK] EvalYOLO ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — Frame loader with stratified temporal sampling
# ═══════════════════════════════════════════════════════════════════════════

def _uniform_indices(N: int, k: int) -> List[int]:
    """
    Return k evenly spaced indices within the range [0, N-1].
    Ensures that both the first and the last frames are always included.
    Copied verbatim from VideoFrameDataset._uniform_indices.
    """
    if k >= N:
        return list(range(N))
    if k == 1:
        return [N // 2]
    indices = sorted(set(
        int(round(i * (N - 1) / (k - 1))) for i in range(k)
    ))
    while len(indices) < k and len(indices) < N:
        gaps  = [(indices[i+1]-indices[i], i) for i in range(len(indices)-1)]
        _, mi = max(gaps)
        mid   = (indices[mi] + indices[mi+1]) // 2
        indices.insert(mi + 1, mid)
    return indices[:k]


def load_test_frames(test_img_paths: List[str],
                     img_size: int          = IMG_SIZE,
                     max_frames: int        = 300,
                     device: torch.device   = DEVICE,
                     sample_mode: str       = "uniform"
                     ) -> Tuple[List[torch.Tensor], List[str]]:
    """
    Load frames from a list of paths (sorted by filename, which corresponds to chronological order).
    Returns a tuple containing a list of tensors of shape (C, H, W) with values in [0, 1], 
    and the corresponding list of selected paths.
    """
    if not test_img_paths:
        return [], []

    # Sort in chronological order
    paths = sorted(test_img_paths, key=lambda p: Path(p).name)
    N     = len(paths)

    if sample_mode == "uniform" and max_frames < N:
        indices = _uniform_indices(N, max_frames)
    else:
        indices = list(range(min(max_frames, N)))

    frames, selected_paths = [], []
    for idx in indices:
        img = cv2.imread(paths[idx])
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        t   = torch.from_numpy(img).permute(2,0,1).float() / 255.0
        frames.append(t.to(device))
        selected_paths.append(paths[idx])

    return frames, selected_paths


print("[OK] load_test_frames + _uniform_indices defined.")
print("     Stratified temporal sampling ensures 100% coverage of the timeline.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — Per-frame evaluation engine (Hungarian Matching)
#
# STANDARD METRIC DEFINITIONS:
#   advBR       = Σ n_adv / Σ n_clean
#                 Box survival ratio across the entire test set ↓ (0 = all boxes vanished)
#
#   IoU_acc     = Σ_frames[ Σ_{matched pairs} IoU(i,j) ] / T
#                 Accumulated matched IoU per frame ↓
#                 Measures the degree of bounding box misalignment or disappearance after the attack
#                 (≠ na/nc — that is the box ratio, not IoU)
#
#   mIoU_matched= Σ_{all matched pairs} IoU(i,j) / n_matched_pairs
#                 Mean IoU per matched pair ↓ (COCO-style standard)
#
#   attack_rate = |{frame f : fn_f > 0}| / |{frame f : nc_f > 0}|
#                 Ratio of effectively attacked frames (losing ≥1 box) ↑
#
#   vanish_rate = |{frame f : na_f=0 ∧ nc_f>0}| / |{frame f : nc_f>0}|
#                 Ratio of frames where ALL boxes completely disappear ↑
#
#   conf_drop   = (1 - μ_conf_adv / μ_conf_clean) × 100  [%]
#                 Relative confidence drop ↑
#
#   MAP         = mean(|δ|)     Stealthiness ↓
#   nuc_norm    = Σ_c ||δ_c||_* Low-rank measure ↓
#
# Matching: Hungarian algorithm (linear_sum_assignment) on the actual IoU matrix
# (box_iou from torchvision) — not just the na/nc ratio.
# ═══════════════════════════════════════════════════════════════════════════

from scipy.optimize import linear_sum_assignment
from torchvision.ops import box_iou as tv_box_iou

def evaluate_single_pair(
        video_name    : str,
        class_mode    : str,
        delta         : torch.Tensor,
        test_frames   : List[torch.Tensor],
        frame_paths   : List[str],
        yolo_wrapper  : EvalYOLO,
        conf_threshold: float = CONF_THRESHOLD,
        iou_match_thr : float = 0.5,          # IoU threshold to consider a pair as matched
        max_viz       : int   = MAX_VIZ_FRAMES,
        output_dir    : Path  = OUTPUT_DIR,
) -> Dict:
    """
    Evaluate a single (video, class) pair across all test frames.
    Utilizes Hungarian matching to compute the true IoU_acc.
    """
    target_cls = CLASS_MODE_TO_YOLO[class_mode]

    # ── Output directories ─────────────────────────────────────────────
    pair_dir  = output_dir / f"{video_name}__{class_mode}"
    dir_clean = pair_dir / "clean"
    dir_adv   = pair_dir / "adversarial"
    dir_diff  = pair_dir / "diff"
    for d in [dir_clean, dir_adv, dir_diff]:
        d.mkdir(parents=True, exist_ok=True)

    N_frames = len(test_frames)
    viz_idx  = set(_uniform_indices(N_frames, min(max_viz, N_frames)))

    # ── Accumulators ───────────────────────────────────────────────────
    per_frame           : List[Dict] = []
    total_clean         = 0          # Σ n_clean (after NMS)
    total_adv           = 0          # Σ n_adv   (after NMS)
    total_matched       = 0          # Σ matched pairs (IoU ≥ thr)
    total_iou_sum       = 0.0        # Σ IoU of matched pairs (for IoU_acc & mIoU)
    frames_attacked     = 0          # |{f : fn_f > 0}|
    frames_vanish       = 0          # |{f : na_f=0 ∧ nc_f>0}|
    frames_no_obj       = 0          # |{f : nc_f=0}|
    conf_clean_all      : List[float] = []
    conf_adv_all        : List[float] = []

    for i, (x_clean, fpath) in enumerate(
            tqdm(zip(test_frames, frame_paths),
                 total=N_frames,
                 desc=f"  Eval {video_name}|{class_mode}")):

        x_adv = torch.clamp(x_clean + delta, 0.0, 1.0)

        clean_boxes = yolo_wrapper.get_boxes(x_clean, target_cls, conf_threshold)
        adv_boxes   = yolo_wrapper.get_boxes(x_adv,   target_cls, conf_threshold)
        nc = len(clean_boxes)
        na = len(adv_boxes)

        conf_clean_all.extend([b["conf"] for b in clean_boxes])
        conf_adv_all.extend(  [b["conf"] for b in adv_boxes])

        total_clean += nc
        total_adv   += na

        # ── Hungarian IoU Matching ─────────────────────────────────────
        # fn = False Negatives: clean boxes that are missed or mislocalized
        frame_iou_sum = 0.0
        matched_this  = 0

        if nc > 0 and na > 0:
            c_t = torch.tensor([b["xyxy"] for b in clean_boxes],
                               dtype=torch.float32)
            a_t = torch.tensor([b["xyxy"] for b in adv_boxes],
                               dtype=torch.float32)
            # tv_box_iou: (nc, na) actual IoU matrix
            iou_mat              = tv_box_iou(c_t, a_t)
            row_ind, col_ind     = linear_sum_assignment(
                                       -iou_mat.cpu().numpy())

            for r, c in zip(row_ind, col_ind):
                iou_val = float(iou_mat[r, c])
                if iou_val >= iou_match_thr:
                    matched_this  += 1
                    frame_iou_sum += iou_val

        total_matched   += matched_this
        total_iou_sum   += frame_iou_sum

        # fn = number of clean boxes NOT matched (suppressed or mislocalized)
        fn = max(0, nc - matched_this)

        if nc == 0:
            frames_no_obj += 1
        else:
            if fn > 0:
                frames_attacked += 1
            if na == 0:
                frames_vanish += 1

        per_frame.append({
            "frame_idx"    : i,
            "frame_name"   : Path(fpath).name,
            "n_clean"      : nc,
            "n_adv"        : na,
            "matched"      : matched_this,
            # suppressed = fn = clean boxes with no matching adversarial box (IoU ≥ thr)
            "suppressed"   : fn,
            # suppress_rate = fn / nc  (ratio of clean boxes suppressed in this frame)
            "suppress_rate": fn / max(nc, 1),
            "all_vanished" : (nc > 0 and na == 0),
            "frame_iou_sum": frame_iou_sum,
            "conf_clean"   : float(np.mean([b["conf"] for b in clean_boxes]))
                             if clean_boxes else 0.0,
            "conf_adv"     : float(np.mean([b["conf"] for b in adv_boxes]))
                             if adv_boxes else 0.0,
        })

        # ── Save annotated frames ───────────────────────────────────────
        if i in viz_idx:
            fn_txt = (f"ALL VANISHED ✓" if na == 0 and nc > 0
                      else f"fn={fn}/{nc} suppressed")
            ann_c = yolo_wrapper.get_annotated_image(
                x_clean, clean_boxes,
                label     = f"CLEAN | boxes={nc}",
                color_bgr = (0, 200, 0))
            ann_a = yolo_wrapper.get_annotated_image(
                x_adv, adv_boxes,
                label     = f"ADV | boxes={na} | {fn_txt}",
                color_bgr = (0, 80, 255))

            diff_np = np.abs(
                (x_clean.cpu().numpy().transpose(1,2,0)*255).astype(np.float32) -
                (x_adv.cpu().numpy().transpose(1,2,0)*255).astype(np.float32))
            diff_norm  = (diff_np.max(axis=2) /
                          max(diff_np.max(), 1e-6) * 255).astype(np.uint8)
            diff_color = cv2.applyColorMap(diff_norm, cv2.COLORMAP_HOT)

            fname = f"frame_{i:04d}_nc{nc}_na{na}_fn{fn}.jpg"
            cv2.imwrite(str(dir_clean / fname), ann_c)
            cv2.imwrite(str(dir_adv   / fname), ann_a)
            cv2.imwrite(str(dir_diff  / fname), diff_color)

    # ══════════════════════════════════════════════════════════════════
    # AGGREGATE METRICS — complete and strictly defined formulas
    # ══════════════════════════════════════════════════════════════════
    T            = len(per_frame)
    frames_w_obj = T - frames_no_obj

    # advBR = Σ n_adv / Σ n_clean  (box survival ratio)
    adv_br = total_adv / max(total_clean, 1)

    # IoU_acc = Σ_frames frame_iou_sum / T
    # = average of the sum of IoU of matched pairs per frame
    # ↓ better: 0 means no adversarial box matches any clean box
    iou_acc = total_iou_sum / max(T, 1)

    # mIoU_matched = Σ IoU / n_matched_pairs  (COCO per-pair standard)
    # ↓ better: remaining adversarial boxes are mislocalized
    miou_matched = total_iou_sum / max(total_matched, 1)

    attack_rate = frames_attacked / max(frames_w_obj, 1)
    vanish_rate = frames_vanish   / max(frames_w_obj, 1)

    mean_cc       = float(np.mean(conf_clean_all)) if conf_clean_all else 0.0
    mean_ca       = float(np.mean(conf_adv_all))   if conf_adv_all   else 0.0
    conf_drop_pct = (1.0 - mean_ca / max(mean_cc, 1e-8)) * 100.0

    map_val  = delta.abs().mean().item()
    nuc_norm = sum(torch.linalg.svdvals(delta[c]).sum().item()
                   for c in range(delta.shape[0]))
    invisible_pct = (delta.abs() < 8/255).float().mean().item() * 100.0

    return {
        "video_name"      : video_name,
        "class_mode"      : class_mode,
        # Counts
        "n_test_frames"   : T,
        "frames_with_obj" : frames_w_obj,
        "frames_no_obj"   : frames_no_obj,
        "total_clean"     : total_clean,
        "total_adv"       : total_adv,
        "suppressed"      : total_clean - total_adv,
        "total_matched"   : total_matched,
        # ── Paper-compatible metrics ───────────────────────────────
        # advBR:    Σ n_adv / Σ n_clean              ↓ better
        "advBR"           : round(adv_br,       6),
        # IoU_acc:  Σ_f Σ_matched IoU / T            ↓ better
        "IoU_acc"         : round(iou_acc,      6),
        # ── Rigorous auxiliary metrics ─────────────────────────────
        # mIoU_matched: Σ IoU / n_pairs              ↓ better
        "mIoU_matched"    : round(miou_matched,  6),
        # attack_rate: |{fn>0}|/|{nc>0}|             ↑ better
        "attack_rate"     : round(attack_rate,   4),
        # vanish_rate: |{na=0 ∧ nc>0}|/|{nc>0}|     ↑ better
        "vanish_rate"     : round(vanish_rate,   4),
        # conf_drop: (1 - μ_adv/μ_clean)×100 [%]    ↑ better
        "conf_drop_pct"   : round(conf_drop_pct, 2),
        # Stealthiness
        "MAP"             : round(map_val,        6),  # ↓
        "nuc_norm"        : round(nuc_norm,       4),  # ↓
        "invisible_pct"   : round(invisible_pct,  2),
        # Raw lists for visualization
        "per_frame"       : per_frame,
        "conf_clean_list" : conf_clean_all,
        "conf_adv_list"   : conf_adv_all,
    }


print("[OK] evaluate_single_pair (RIGOROUS v2 — Hungarian IoU Matching)")
print()
print("  Metric formulas:")
print("  advBR        = Σ n_adv / Σ n_clean                      ↓")
print("  IoU_acc      = Σ_frames[Σ_matched IoU(i,j)] / T         ↓")
print("  mIoU_matched = Σ IoU(matched pairs) / n_pairs            ↓")
print("  attack_rate  = |{f: fn>0}| / |{f: nc>0}|                ↑")
print("  vanish_rate  = |{f: na=0∧nc>0}| / |{f: nc>0}|           ↑")
print("  conf_drop    = (1 - μ_conf_adv/μ_conf_clean) × 100 [%]  ↑")
print("  MAP          = mean(|δ|)                                  ↓")
print("  nuc_norm     = Σ_c ||δ_c||_*                             ↓")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — Scientific 4-panel visualization 
# ═══════════════════════════════════════════════════════════════════════════

COLOR_CLEAN = "#2ecc71"    #clean detection
COLOR_ADV   = "#e74c3c"    #adversarial 
COLOR_SUPP  = "#e67e22"    #suppressed
FONT_TITLE  = 12
FONT_LABEL  = 10
FONT_TICK   = 9

plt.rcParams.update({
    "font.family"    : "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "grid.linestyle" : "--",
})


def plot_4panel_statistics(
        result      : Dict,
        save_path   : Path,
        dpi         : int = 150,
) -> None:
    """
    4-panel figure 
    Panel 1: Box count theo timeline (area chart Clean vs Adv)
    Panel 2: Suppress rate per frame (%)
    Panel 3: Confidence score distribution (histogram)
    Panel 4: Box count per frame distribution (histogram)
    """
    per_frame = result["per_frame"]
    if not per_frame:
        print("  [WARN] No per-frame data to plot.")
        return

    video   = result["video_name"]
    cls     = result["class_mode"]
    advBR   = result["advBR"]
    iou_acc = result["IoU_acc"]
    vrate   = result["vanish_rate"] * 100
    cdrop   = result["conf_drop_pct"]

    frames       = [r["frame_idx"]     for r in per_frame]
    n_clean      = [r["n_clean"]       for r in per_frame]
    n_adv        = [r["n_adv"]         for r in per_frame]
    supp_rate    = [r["suppress_rate"] * 100 for r in per_frame]
    conf_clean   = result["conf_clean_list"]
    conf_adv     = result["conf_adv_list"]

    fig = plt.figure(figsize=(16, 11))
    fig.patch.set_facecolor("white")

    # ── Suptitle ──────────────────────────────
    fig.suptitle(
        f"UAP Attack Effectiveness Analysis\n"
        f"Video: {video}  |  Target Class: {cls.upper()}  |  "
        f"Model: YOLOv11m (COCO pretrained)\n"
        f"advBR={advBR:.4f} ↓  |  IoU_acc={iou_acc:.4f} ↓  |  "
        f"Vanish={vrate:.1f}%  |  Conf Drop={cdrop:.1f}%",
        fontsize=FONT_TITLE, fontweight="bold", y=0.98,
        color="#2c3e50"
    )

    gs = gridspec.GridSpec(2, 2, figure=fig,
                           hspace=0.42, wspace=0.32,
                           left=0.07, right=0.97,
                           top=0.88, bottom=0.08)

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Panel 1: Detection count through timeline
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.fill_between(frames, n_clean, color=COLOR_CLEAN,
                     alpha=0.40, label="Clean detection")
    ax1.fill_between(frames, n_adv,   color=COLOR_ADV,
                     alpha=0.65, label="Adversarial detection")
    ax1.plot(frames, n_clean, color=COLOR_CLEAN, lw=1.2, alpha=0.8)
    ax1.plot(frames, n_adv,   color=COLOR_ADV,   lw=1.2, alpha=0.9)

    # Annotate mean lines
    mc = np.mean(n_clean) if n_clean else 0
    ma = np.mean(n_adv)   if n_adv   else 0
    ax1.axhline(mc, color=COLOR_CLEAN, lw=1.0, ls=":", alpha=0.9,
                label=f"Mean clean={mc:.1f}")
    ax1.axhline(ma, color=COLOR_ADV,   lw=1.0, ls=":", alpha=0.9,
                label=f"Mean adv={ma:.1f}")

    ax1.set_title("① Bounding Box Count Along Timeline",
                  fontsize=FONT_TITLE, fontweight="bold", pad=8)
    ax1.set_xlabel("Frame Index (chronological order)", fontsize=FONT_LABEL)
    ax1.set_ylabel("# Detected Bounding Boxes (NMS)", fontsize=FONT_LABEL)
    ax1.tick_params(labelsize=FONT_TICK)
    ax1.legend(fontsize=8, loc="upper right", framealpha=0.8)

    total_supp = result["total_clean"] - result["total_adv"]
    ax1.text(0.02, 0.97,
             f"Total suppressed: {total_supp}/{result['total_clean']} "
             f"({(1-advBR)*100:.1f}%)",
             transform=ax1.transAxes, fontsize=8,
             va="top", color="#7f0000",
             bbox=dict(boxstyle="round,pad=0.3", fc="#fff3cd", ec="#e0a800"))

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Panel 2: Suppress rate per frame
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ax2 = fig.add_subplot(gs[0, 1])

    # Color-code bars: đỏ đậm = 100% vanish, cam = partial, xanh = không attack
    bar_colors = []
    for r in per_frame:
        if r["all_vanished"]:
            bar_colors.append("#c0392b")          # đỏ đậm
        elif r["suppress_rate"] > 0.5:
            bar_colors.append(COLOR_SUPP)          # cam
        elif r["suppress_rate"] > 0:
            bar_colors.append("#f39c12")           # vàng
        else:
            bar_colors.append("#95a5a6")           # abu — không hiệu quả

    ax2.bar(frames, supp_rate, color=bar_colors, alpha=0.85, width=1.0)
    ax2.axhline(100, color="#2c3e50", lw=1.2, ls="--", alpha=0.6,
                label="100% = all objects vanished")
    mean_sr = np.mean([r["suppress_rate"] for r in per_frame
                       if r["n_clean"] > 0]) * 100
    ax2.axhline(mean_sr, color="#8e44ad", lw=1.5, ls="-.",
                label=f"Mean suppress={mean_sr:.1f}%")

    ax2.set_title("② Per-Frame Suppress Rate (%)",
                  fontsize=FONT_TITLE, fontweight="bold", pad=8)
    ax2.set_xlabel("Frame Index", fontsize=FONT_LABEL)
    ax2.set_ylabel("Suppress Rate (%)", fontsize=FONT_LABEL)
    ax2.set_ylim(0, 115)
    ax2.tick_params(labelsize=FONT_TICK)
    ax2.legend(fontsize=8, framealpha=0.8)

    # Legend màu sắc
    patches_legend = [
        mpatches.Patch(color="#c0392b", label="100% vanished"),
        mpatches.Patch(color=COLOR_SUPP, label=">50% suppressed"),
        mpatches.Patch(color="#f39c12",  label="<50% suppressed"),
        mpatches.Patch(color="#95a5a6",  label="Not effective"),
    ]
    ax2.legend(handles=patches_legend, fontsize=7.5,
               loc="upper left", framealpha=0.85)

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Panel 3: Confidence score distribution
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ax3 = fig.add_subplot(gs[1, 0])

    bins = np.linspace(0, 1, 41)
    if conf_clean:
        ax3.hist(conf_clean, bins=bins, color=COLOR_CLEAN, alpha=0.55,
                 label=f"Clean (n={len(conf_clean)})", density=True)
        ax3.axvline(np.mean(conf_clean), color=COLOR_CLEAN,
                    lw=2, ls="--",
                    label=f"μ_clean={np.mean(conf_clean):.3f}")
    if conf_adv:
        ax3.hist(conf_adv,   bins=bins, color=COLOR_ADV,   alpha=0.70,
                 label=f"Adversarial (n={len(conf_adv)})", density=True)
        ax3.axvline(np.mean(conf_adv), color=COLOR_ADV,
                    lw=2, ls="--",
                    label=f"μ_adv={np.mean(conf_adv):.3f}")
    ax3.axvline(CONF_THRESHOLD, color="#2c3e50", lw=1.2, ls=":",
                label=f"τ={CONF_THRESHOLD}")

    ax3.set_title("③ Detection Confidence Score Distribution",
                  fontsize=FONT_TITLE, fontweight="bold", pad=8)
    ax3.set_xlabel("Confidence Score", fontsize=FONT_LABEL)
    ax3.set_ylabel("Density", fontsize=FONT_LABEL)
    ax3.tick_params(labelsize=FONT_TICK)
    ax3.legend(fontsize=8, framealpha=0.8)

    # Annotate conf drop
    ax3.text(0.98, 0.97,
             f"Conf drop: {cdrop:.1f}%\n"
             f"Remaining dets: {len(conf_adv)}/{max(len(conf_clean),1)} "
             f"({len(conf_adv)/max(len(conf_clean),1)*100:.1f}%)",
             transform=ax3.transAxes, fontsize=8,
             ha="right", va="top", color="#7f0000",
             bbox=dict(boxstyle="round,pad=0.3", fc="#fdecea", ec=COLOR_ADV))

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Panel 4: Box count per frame distribution
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ax4 = fig.add_subplot(gs[1, 1])

    max_boxes = max(max(n_clean + [1]), max(n_adv + [1]))
    bins_b    = np.arange(0, max_boxes + 2) - 0.5
    ax4.hist(n_clean, bins=bins_b, color=COLOR_CLEAN,
             alpha=0.55, label=f"Clean  μ={np.mean(n_clean):.2f}")
    ax4.hist(n_adv,   bins=bins_b, color=COLOR_ADV,
             alpha=0.70, label=f"Adv    μ={np.mean(n_adv):.2f}")

    ax4.set_title("④ Distribution of Boxes per Frame",
                  fontsize=FONT_TITLE, fontweight="bold", pad=8)
    ax4.set_xlabel("# Bounding Boxes per Frame", fontsize=FONT_LABEL)
    ax4.set_ylabel("# Frames", fontsize=FONT_LABEL)
    ax4.tick_params(labelsize=FONT_TICK)
    ax4.legend(fontsize=9, framealpha=0.8)

    # Annotate vanish frames
    n_vanish = sum(1 for r in per_frame if r["all_vanished"])
    ax4.text(0.98, 0.97,
             f"Frames all-vanished: {n_vanish}\n"
             f"({n_vanish/max(result['frames_with_obj'],1)*100:.1f}% "
             f"of frames w/ objects)",
             transform=ax4.transAxes, fontsize=8,
             ha="right", va="top",
             bbox=dict(boxstyle="round,pad=0.3", fc="#fdecea", ec=COLOR_ADV))

    # ── Footnote nghiên cứu ───────────────────────────────────────────
    fig.text(0.5, 0.005,
             "AO-Exp Attack (arXiv:2510.14460v1) | "
             "Universal Adversarial Perturbation on YOLOv11m | "
             f"δ: MAP={result['MAP']:.5f}, ‖δ‖_*={result['nuc_norm']:.2f}, "
             f"invis={result['invisible_pct']:.1f}%",
             ha="center", fontsize=7.5, color="#7f8c8d",
             style="italic")

    plt.savefig(str(save_path), dpi=dpi, bbox_inches="tight",
                facecolor="white")
    plt.close(fig)
    print(f"  [VIZ] Saved 4-panel → {save_path.name}")


print("[OK] plot_4panel_statistics defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Perturbation quality visualization (delta heatmap + spectrum)
# ═══════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Perturbation quality visualization (Revised: 3 Panels)
# Panel 1: Delta Heatmap
# Panel 2: Clean Detection (with BBoxes)
# Panel 3: Adversarial Detection (with BBoxes)
# ═══════════════════════════════════════════════════════════════════════════
def plot_perturbation_quality(
    delta      : torch.Tensor,    # (3, H, W)
    result     : Dict,
    sample_frame: Optional[torch.Tensor],  # (3,H,W) hoặc None
    save_path  : Path,
    yolo_wrapper: EvalYOLO = None,  
    target_cls : set = None,      
    conf_thr   : float = CONF_THRESHOLD,
    dpi        : int = 150,
) -> None:
    delta_np = delta.cpu().numpy()            # (3, H, W)
    mag      = np.abs(delta_np).max(axis=0)   # (H, W)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="white")
    fig.suptitle(
        f"UAP Perturbation Quality  |  "
        f"{result['video_name']} | class={result['class_mode']}\n"
        f"MAP={result['MAP']:.5f}  ‖δ‖_*={result['nuc_norm']:.2f}  "
        f"invis={result['invisible_pct']:.1f}% (|δ|<8/255)",
        fontsize=11, fontweight="bold", y=0.98
    )

    # ── Panel 1: Magnitude heatmap───────────────────────
    ax = axes[0]
    im = ax.imshow(mag, cmap="hot", vmin=0, vmax=max(float(mag.max()), 1e-6))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Max |δ| over RGB channels")
    ax.set_title("Perturbation Magnitude\n(max over RGB channels)", fontsize=10)
    ax.axis("off")
    ax.text(0.02, 0.02,
            f"δ range: [{delta.min():.3f}, {delta.max():.3f}]\n"
            f"≥8/255: {100-result['invisible_pct']:.1f}% pixels",
            transform=ax.transAxes, fontsize=8, color="white",
            va="bottom",
            bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.6))

    # ─ Panel 2 & 3: Detection Visualization ──────────────────────────
    if sample_frame is not None and yolo_wrapper is not None and target_cls is not None:
        # Tạo ảnh Adv
        x_adv = torch.clamp(sample_frame + delta, 0.0, 1.0)
        
        # --- Panel 2: Clean ---
        clean_boxes = yolo_wrapper.get_boxes(sample_frame, target_cls, conf_thr)
        img_clean = yolo_wrapper.get_annotated_image(
            sample_frame, clean_boxes, 
            label=f"CLEAN | boxes={len(clean_boxes)}", 
            color_bgr=(0, 200, 0) # Màu xanh lá
        )
        axes[1].imshow(cv2.cvtColor(img_clean, cv2.COLOR_BGR2RGB))
        axes[1].set_title("Clean Detection", fontsize=10, fontweight="bold")
        axes[1].axis("off")
        
        # --- Panel 3: Adversarial ---
        adv_boxes = yolo_wrapper.get_boxes(x_adv, target_cls, conf_thr)
        img_adv = yolo_wrapper.get_annotated_image(
            x_adv, adv_boxes,
            label=f"ADV | boxes={len(adv_boxes)}", 
            color_bgr=(0, 80, 255) 
        )
        axes[2].imshow(cv2.cvtColor(img_adv, cv2.COLOR_BGR2RGB))
        axes[2].set_title("Adversarial Detection", fontsize=10, fontweight="bold")
        axes[2].axis("off")
        
        axes[2].text(0.05, 0.05, 
                     f"Suppressed: {len(clean_boxes) - len(adv_boxes)} boxes",
                     transform=axes[2].transAxes, fontsize=10, fontweight="bold",
                     color="red", va="bottom",
                     bbox=dict(boxstyle="round", fc="white", alpha=0.8))
    else:
        axes[1].text(0.5, 0.5, "No wrapper provided", ha="center", va="center")
        axes[1].axis("off")
        axes[2].text(0.5, 0.5, "No wrapper provided", ha="center", va="center")
        axes[2].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(str(save_path), dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [VIZ] Saved perturbation quality → {save_path.name}")

print("[OK] plot_perturbation_quality (REVISED) defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9 — EXCEL EXPORT (MULTI-SHEET): Summary | Per-Video | Per-Frame Details
# ═══════════════════════════════════════════════════════════════════════════

def _cell_style(ws, row, col, value,
                bold=False, fill_hex=None, align="center",
                num_fmt=None, font_color="000000"):
    cell = ws.cell(row=row, column=col, value=value)
    cell.font      = Font(bold=bold, color=font_color, size=10)
    cell.alignment = Alignment(horizontal=align, vertical="center")
    if fill_hex:
        cell.fill = PatternFill("solid", fgColor=fill_hex)
    if num_fmt:
        cell.number_format = num_fmt
    thin = Side(style="thin")
    cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)
    return cell


def export_excel(all_results: List[Dict], save_path: Path) -> None:
    if not XLSX_OK:
        print("  [WARN] openpyxl không có — bỏ qua Excel export.")
        return

    wb = openpyxl.Workbook()

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # SHEET 1 — Summary
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ws1 = wb.active
    ws1.title = "Summary"

    # Title header
    ws1.merge_cells("A1:N1")
    title_cell = ws1["A1"]
    title_cell.value = (
        "UAP Evaluation Summary — Structured Universal Adversarial Attacks "
        "on YOLOv11m  |  arXiv:2510.14460v1"
    )
    title_cell.font      = Font(bold=True, size=12, color="FFFFFF")
    title_cell.fill      = PatternFill("solid", fgColor="2C3E50")
    title_cell.alignment = Alignment(horizontal="center", vertical="center")
    ws1.row_dimensions[1].height = 24

    headers = [
        "Video", "Class",
        "Test Frames", "Frames w/ Obj",
        "Clean Boxes (NMS)", "Adv Boxes (NMS)",
        "Suppressed", "Suppress %",
        # ── Paper metrics ──────────────────────────────────────────
        "advBR ↓",          # Σ n_adv / Σ n_clean
        "IoU_acc ↓",        # Σ_f[Σ matched IoU] / T
        "mIoU_matched ↓",   # Σ IoU / n_pairs  (NEW — chuẩn COCO)
        # ── Effectiveness ──────────────────────────────────────────
        "Attack Rate ↑",    # |{fn>0}|/|{nc>0}|
        "Vanish Rate ↑",    # |{na=0∧nc>0}|/|{nc>0}|
        "Conf Drop % ↑",    # (1-μ_adv/μ_clean)×100
        # ── Stealthiness ───────────────────────────────────────────
        "MAP ↓",
        "‖δ‖_*",
        "Invis %",
    ]
    HDR_FILL  = "2980B9"
    META_FILL = "D5E8F7"
    GOOD_FILL = "D5F5E3"
    WARN_FILL = "FDECEA"

    for ci, h in enumerate(headers, 1):
        _cell_style(ws1, 2, ci, h,
                    bold=True, fill_hex=HDR_FILL, font_color="FFFFFF")

    for ri, res in enumerate(all_results, 3):
        supp_pct = (res["suppressed"] / max(res["total_clean"],1)) * 100
        row_data = [
            res["video_name"], res["class_mode"],
            res["n_test_frames"], res["frames_with_obj"],
            res["total_clean"], res["total_adv"],
            res["suppressed"], round(res["suppressed"]/max(res["total_clean"],1)*100, 1),
            res["advBR"],
            res["IoU_acc"],
            res["mIoU_matched"],        # NEW
            res["attack_rate"],
            res["vanish_rate"],
            res["conf_drop_pct"],
            res["MAP"],
            res["nuc_norm"],
            res["invisible_pct"],
        ]
        fill = META_FILL
        for ci, val in enumerate(row_data, 1):
            fmt = None
            if ci in (9, 10):       fmt = "0.0000"
            elif ci in (11, 12):    fmt = "0.00%"
            elif ci == 14:          fmt = "0.000000"
            _cell_style(ws1, ri, ci, val,
                        fill_hex=fill, num_fmt=fmt, align="center")

    # Auto column width
    for col in ws1.columns:
        max_w = max(len(str(c.value or "")) for c in col) + 3
        ws1.column_dimensions[get_column_letter(col[0].column)].width = min(max_w, 22)

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # SHEET 2 — Comparison Chart Data
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    ws2 = wb.create_sheet("Metrics Comparison")
    comp_headers = ["Pair", "advBR", "IoU_acc",
                    "Attack Rate", "Vanish Rate", "Conf Drop %"]
    for ci, h in enumerate(comp_headers, 1):
        _cell_style(ws2, 1, ci, h,
                    bold=True, fill_hex=HDR_FILL, font_color="FFFFFF")

    for ri, res in enumerate(all_results, 2):
        pair = f"{res['video_name']} | {res['class_mode']}"
        row  = [pair, res["advBR"], res["IoU_acc"],
                res["attack_rate"], res["vanish_rate"],
                res["conf_drop_pct"]]
        for ci, val in enumerate(row, 1):
            _cell_style(ws2, ri, ci, val, align="center")

    # Bar chart — advBR per pair
    if len(all_results) >= 2:
        chart = BarChart()
        chart.type    = "col"
        chart.title   = "advBR per (Video, Class) — lower is better"
        chart.y_axis.title = "advBR (Box Survival Ratio)"
        chart.x_axis.title = "Attack Pair"
        chart.shape   = 4
        chart.width   = 20
        chart.height  = 12

        data_ref = Reference(ws2,
                             min_col=2, max_col=2,
                             min_row=1,
                             max_row=1 + len(all_results))
        cats_ref = Reference(ws2,
                             min_col=1,
                             min_row=2,
                             max_row=1 + len(all_results))
        chart.add_data(data_ref, titles_from_data=True)
        chart.set_categories(cats_ref)
        ws2.add_chart(chart, "H2")

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # SHEET 3+ — Per-Frame detail (1 sheet per pair, max 11 pairs)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    pf_headers = ["Frame", "n_clean", "n_adv", "Suppressed",
                  "Suppress%", "All Vanished", "Conf Clean", "Conf Adv"]

    for res in all_results:
        sheet_name = f"{res['video_name'][:14]}_{res['class_mode']}"
        ws_pf = wb.create_sheet(sheet_name)

        for ci, h in enumerate(pf_headers, 1):
            _cell_style(ws_pf, 1, ci, h,
                        bold=True, fill_hex=HDR_FILL, font_color="FFFFFF")

        for ri, pf in enumerate(res["per_frame"], 2):
            supp_pct = round(pf["suppress_rate"] * 100, 1)
            row = [
                pf["frame_name"], pf["n_clean"], pf["n_adv"],
                pf["suppressed"], supp_pct,
                "YES" if pf["all_vanished"] else "",
                round(pf["conf_clean"], 4), round(pf["conf_adv"], 4),
            ]
            fill_r = ("FFD6CC" if pf["all_vanished"]
                      else ("FFF3CD" if pf["suppressed"] > 0 else None))
            for ci, val in enumerate(row, 1):
                _cell_style(ws_pf, ri, ci, val,
                            fill_hex=fill_r, align="center")

        # Auto width
        for col in ws_pf.columns:
            max_w = max(len(str(c.value or "")) for c in col) + 2
            ws_pf.column_dimensions[
                get_column_letter(col[0].column)].width = min(max_w, 20)

    wb.save(str(save_path))
    print(f"  [XLSX] Saved → {save_path.name}  "
          f"({len(wb.sheetnames)} sheets)")


print("[OK] export_excel defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10 — MAIN: Batch evaluation all UAP files
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "═"*62)
print("  [STEP 4] BATCH EVALUATION — All Found UAPs")
print("═"*62)

all_results: List[Dict] = []

for (video_name, class_mode), pert_path in sorted(uap_map.items()):
    print(f"\n{'─'*62}")
    print(f"  UAP: {pert_path.name}")
    print(f"  Pair: {video_name} | {class_mode}")
    print(f"{'─'*62}")

    # ── 1. Load UAP ────────────────────────────────────────────────
    delta, meta = load_uap(pert_path, DEVICE)
    hp = meta["hyperparams"]
    print(f"  [META] λ₁={hp.get('lambda1')} λ₂={hp.get('lambda2')} "
          f"k={hp.get('k_rank')} T={hp.get('n_iter')} "
          f"@ {meta['timestamp']}")
    print(f"  [META] Train metrics: "
          f"advBR={meta['metrics'].get('advBR','?')} "
          f"IoU_acc={meta['metrics'].get('IoU_acc','?')}")

    # ── 2. Load test frames (test.txt) ─────
    test_paths = video_test_files.get(video_name, [])
    if not test_paths:
        print(f"  [WARN] Không có test frames cho {video_name} → skip")
        continue

    # Recommend max_frames theo video size
    total_frames = VIDEO_FRAME_COUNTS.get(video_name, 912)
    max_f = min(total_frames, 300) if total_frames <= 1000 else \
            min(total_frames, 400) if total_frames <= 2000 else \
            min(total_frames, 450) if total_frames <= 3000 else 500

    test_frames, frame_paths = load_test_frames(
        test_paths, img_size=IMG_SIZE,
        max_frames=max_f, device=DEVICE,
        sample_mode="uniform"
    )
    print(f"  [LOAD] {len(test_frames)} test frames loaded "
          f"(from {len(test_paths)} available)")

    # ── 3. Evaluate ────────────────────────────────────────────────
    pair_output = OUTPUT_DIR / f"{video_name}__{class_mode}"
    result = evaluate_single_pair(
        video_name     = video_name,
        class_mode     = class_mode,
        delta          = delta,
        test_frames    = test_frames,
        frame_paths    = frame_paths,
        yolo_wrapper   = eval_yolo,
        conf_threshold = CONF_THRESHOLD,
        max_viz        = MAX_VIZ_FRAMES,
        output_dir     = OUTPUT_DIR,
    )
    all_results.append(result)

    # ── 4. Print per-pair results ──────────────────────────────────
    supp_pct = result["suppressed"] / max(result["total_clean"],1) * 100
    
    conf_clean_list = result.get("conf_clean_list", [])
    conf_adv_list   = result.get("conf_adv_list", [])
    mean_cc = float(np.mean(conf_clean_list)) if conf_clean_list else 0.0
    mean_ca = float(np.mean(conf_adv_list))   if conf_adv_list   else 0.0
    
    print(f"""
  ┌─ RESULTS: {video_name} | {class_mode} {'─'*20}
  │ Test frames      : {result['n_test_frames']}
  │ Frames w/ object : {result['frames_with_obj']}
  │ Clean boxes (NMS): {result['total_clean']}
  │ Adv   boxes (NMS): {result['total_adv']}
  │ Suppressed       : {result['suppressed']} ({supp_pct:.1f}%)
  │ advBR            : {result['advBR']:.4f}  ↓ (0=all vanished)
  │ IoU_acc          : {result['IoU_acc']:.4f}  ↓
  │ mIoU_matched     : {result['mIoU_matched']:.4f}  ↓ (NEW)
  │ Attack rate      : {result['attack_rate']*100:.1f}%  ↑
  │ Vanish rate      : {result['vanish_rate']*100:.1f}%  ↑
  │ Conf: clean={mean_cc:.4f} → adv={mean_ca:.4f}
  │       drop={result['conf_drop_pct']:.1f}%
  │ MAP              : {result['MAP']:.6f}  ↓
  │ Nuclear norm ‖δ‖_*: {result['nuc_norm']:.4f}
  │ Invisible pixels : {result['invisible_pct']:.1f}%
  └{'─'*52}""")

    # ── 5. Visualizations ──────────────────────────────────────────
    plot_4panel_statistics(
        result,
        save_path = pair_output / "statistics_4panel.png",
    )
    
    sample_frame = test_frames[len(test_frames)//2] if test_frames else None
    
    plot_perturbation_quality(
        delta          = delta, 
        result         = result, 
        sample_frame   = sample_frame,
        save_path      = pair_output / "perturbation_quality.png",
        yolo_wrapper   = eval_yolo,
        target_cls     = CLASS_MODE_TO_YOLO[class_mode],
    )

print(f"\n[BATCH] Hoàn tất {len(all_results)} / {len(uap_map)} cặp.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11 — Summary table + Excel + Comparison chart 
# ═══════════════════════════════════════════════════════════════════════════

if not all_results:
    print("[WARN] Không có kết quả.")
else:
    # ── Console summary ────────────────────────────────────────────────
    print("\n" + "═"*92)
    print(f"{'EVALUATION SUMMARY — ALL UAP PAIRS':^92}")
    print(f"{'Paper: AO-Exp Attack (arXiv:2510.14460v1) | Model: YOLOv11m':^92}")
    print("═"*92)
    print(
        f"  {'Video':<28} {'Class':<8}"
        f" {'advBR↓':>7} {'IoU_acc↓':>9} {'mIoU↓':>7}"
        f" {'Atk%↑':>6} {'Vnsh%↑':>7} {'CfDrp%↑':>8}"
        f" {'MAP↓':>9} {'‖δ‖_*':>7}"
    )
    print("─"*92)

    for res in sorted(all_results,
                       key=lambda r: (r["video_name"], r["class_mode"])):
        print(
            f"  {res['video_name']:<28} {res['class_mode']:<8}"
            f" {res['advBR']:>7.4f}"
            f" {res['IoU_acc']:>9.4f}"
            f" {res['mIoU_matched']:>7.4f}"
            f" {res['attack_rate']*100:>5.1f}%"
            f" {res['vanish_rate']*100:>6.1f}%"
            f" {res['conf_drop_pct']:>7.1f}%"
            f" {res['MAP']:>9.5f}"
            f" {res['nuc_norm']:>7.2f}"
        )

    print("─"*92)
    avg = lambda k: np.mean([r[k] for r in all_results])
    print(
        f"  {'AVERAGE':<36}"
        f" {avg('advBR'):>7.4f}"
        f" {avg('IoU_acc'):>9.4f}"
        f" {avg('mIoU_matched'):>7.4f}"
        f" {avg('attack_rate')*100:>5.1f}%"
        f" {avg('vanish_rate')*100:>6.1f}%"
        f" {avg('conf_drop_pct'):>7.1f}%"
        f" {avg('MAP'):>9.5f}"
    )
    print("═"*92)

    # ── Metric legend ──────────────────────────────────────────────────
    print("""
  METRIC LEGEND:
  advBR        Σ n_adv / Σ n_clean           Box survival ratio     ↓ better (0=all vanished)
  IoU_acc      Σ_f[Σ matched IoU] / T        Accum. matched IoU/f   ↓ better (0=perfect attack)
  mIoU         Σ IoU(matched) / n_pairs      Per-pair mean IoU      ↓ better (box lệch vị trí)
  Atk%         |{fn>0}|/|{nc>0}| ×100        Frame attack rate       ↑ better
  Vnsh%        |{na=0∧nc>0}|/|{nc>0}|×100   Frame vanish rate       ↑ better
  CfDrp%       (1-μ_adv/μ_clean)×100         Confidence drop        ↑ better
  MAP          mean(|δ|)                      Perturbation magnitude ↓ stealthier
  ‖δ‖_*        Σ_c svdvals(δ_c).sum()        Nuclear norm           ↓ low-rank
    """)

    # ── Comparison bar chart ───────────────────────────────────────────
    pairs   = [f"{r['video_name'][:16]}\n{r['class_mode']}"
               for r in all_results]
    adv_br  = [r["advBR"]           for r in all_results]
    iou_ac  = [r["IoU_acc"]         for r in all_results]
    miou    = [r["mIoU_matched"]    for r in all_results]
    van_r   = [r["vanish_rate"]     for r in all_results]
    cfdrop  = [r["conf_drop_pct"]/100 for r in all_results]

    x = np.arange(len(pairs))
    w = 0.16

    fig2, axes2 = plt.subplots(1, 2, figsize=(max(14, len(pairs)*2.4), 6),
                                facecolor="white")

    # ── Sub-plot A: advBR / IoU_acc / mIoU ────────────────────────────
    ax = axes2[0]
    b1 = ax.bar(x - w,   adv_br, w, label="advBR ↓",      color="#e74c3c", alpha=0.88)
    b2 = ax.bar(x,       iou_ac, w, label="IoU_acc ↓",    color="#e67e22", alpha=0.88)
    b3 = ax.bar(x + w,   miou,   w, label="mIoU_matched ↓",color="#9b59b6", alpha=0.88)

    ax.set_xticks(x); ax.set_xticklabels(pairs, fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Metric Value", fontsize=10)
    ax.set_title("Suppression & Localization Metrics\n"
                 "(lower = stronger attack)", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3, ls="--")
    for bars in [b1, b2, b3]:
        for rect in bars:
            h = rect.get_height()
            ax.text(rect.get_x()+rect.get_width()/2, h+0.01,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=6.5)

    # ── Sub-plot B: Vanish rate / Conf drop ───────────────────────────
    ax = axes2[1]
    b4 = ax.bar(x - w/2, van_r,  w*1.5, label="Vanish Rate ↑",  color="#2ecc71", alpha=0.88)
    b5 = ax.bar(x + w/2, cfdrop, w*1.5, label="Conf Drop ↑",    color="#3498db", alpha=0.88)

    ax.set_xticks(x); ax.set_xticklabels(pairs, fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Rate (0–1)", fontsize=10)
    ax.set_title("Attack Effectiveness Metrics\n"
                 "(higher = stronger attack)", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3, ls="--")
    for bars in [b4, b5]:
        for rect in bars:
            h = rect.get_height()
            ax.text(rect.get_x()+rect.get_width()/2, h+0.01,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=6.5)

    fig2.suptitle(
        "UAP Attack Effectiveness — All (Video, Class) Pairs\n"
        "AO-Exp Attack | YOLOv11m | arXiv:2510.14460v1",
        fontsize=12, fontweight="bold", y=1.02
    )
    fig2.text(0.5, -0.01,
              "advBR & IoU metrics: lower = better attack  |  "
              "Vanish Rate & Conf Drop: higher = better attack",
              ha="center", fontsize=8.5, color="#555", style="italic")
    plt.tight_layout()

    comp_path = OUTPUT_DIR / "comparison_all_pairs.png"
    plt.savefig(str(comp_path), dpi=140, bbox_inches="tight",
                facecolor="white")
    plt.close(fig2)
    print(f"  [VIZ] Comparison chart → {comp_path.name}")

    # ── Excel export ───────────────────────────────────────────────────
    xlsx_path = OUTPUT_DIR / "uap_evaluation_results.xlsx"
    export_excel(all_results, xlsx_path)

    print(f"\n[DONE] ════════════════════════════════════════════")
    print(f"  Output  → {OUTPUT_DIR}")
    print(f"  Excel   → {xlsx_path.name}")
    print(f"  Pairs   : {len(all_results)}")
    total_supp = sum(r['suppressed'] for r in all_results)
    total_cln  = sum(r['total_clean'] for r in all_results)
    print(f"  Suppressed: {total_supp}/{total_cln} "
          f"({total_supp/max(total_cln,1)*100:.1f}%)")
    print(f"[DONE] ════════════════════════════════════════════")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL A (rewrite) — LAZY FRAME ITERATOR
# ═══════════════════════════════════════════════════════════════════════════

def iter_video_frames(
        dataset_root : str,
        video_name   : str,
        img_size     : int = IMG_SIZE,
        stride       : int = 1,
) -> Tuple[int, any]:
    """
    Generator: yields (frame_index, cpu_tensor) for each frame sequentially.
    Tensors are maintained on the CPU and transferred to the GPU only during inference, 
    then immediately freed to minimize memory usage.
    No frames are kept in memory.
    """
    dataset_root = Path(dataset_root)
    img_dir = dataset_root / video_name / "images"
    if not img_dir.exists():
        img_dir = dataset_root / "images" / video_name
    if not img_dir.exists():
        print(f"  [WARN] Image directory not found: {video_name}")
        return

    all_paths = sorted(
        [p for p in img_dir.iterdir()
         if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}],
        key=lambda p: p.name
    )
    all_paths = all_paths[::stride]

    for idx, p in enumerate(all_paths):
        img = cv2.imread(str(p))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        t   = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        # Yield CPU tensor + path — avoid .to(device) here to save memory
        yield idx, t, str(p)


def get_optimal_stride(video_name: str) -> int:
    n = VIDEO_FRAME_COUNTS.get(video_name, 1000)
    if n <= 1000: return 1
    if n <= 2000: return 2
    return 3


# Avoid pre-loading data — store only metadata
full_video_meta: Dict[str, Dict] = {}
for vname in VIDEO_NAMES:
    stride = get_optimal_stride(vname)
    total  = VIDEO_FRAME_COUNTS.get(vname, 0)
    est_frames = (total + stride - 1) // stride
    full_video_meta[vname] = {"stride": stride, "est_frames": est_frames}
    print(f"  [META] {vname:<30} stride={stride}  ~{est_frames} frames")

print("\n[OK] Lazy iterator ready — zero frames loaded into memory.")
print("     Frames are read, evaluated, and freed on-the-fly during the evaluation loop.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL A — Count ACTUAL frames from the directory (ignore VIDEO_FRAME_COUNTS)
# Note: VIDEO_FRAME_COUNTS represents the original video frame count, 
# NOT the number of annotated frames.
# Number of annotated frames = train + val + test 
# (e.g., annichkov_most_001 = 639 + 137 + 138 = 914)
# ═══════════════════════════════════════════════════════════════════════════

def count_actual_frames(dataset_root: str, video_name: str) -> int:
    """Count the actual number of image files present in the video's image directory."""
    dataset_root = Path(dataset_root)
    img_dir = dataset_root / video_name / "images"
    if not img_dir.exists():
        img_dir = dataset_root / "images" / video_name
    if not img_dir.exists():
        return 0
    return sum(
        1 for p in img_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    )


def iter_video_frames(
        dataset_root : str,
        video_name   : str,
        img_size     : int = IMG_SIZE,
        stride       : int = 1,
):
    """
    Generator: yields (frame_index, cpu_tensor, path) sequentially for each frame.
    Tensors are kept on the CPU and transferred to the GPU only during inference, 
    then immediately freed to minimize memory usage.
    """
    dataset_root = Path(dataset_root)
    img_dir = dataset_root / video_name / "images"
    if not img_dir.exists():
        img_dir = dataset_root / "images" / video_name
    if not img_dir.exists():
        print(f"  [WARN] Image directory not found: {video_name}")
        return

    all_paths = sorted(
        [p for p in img_dir.iterdir()
         if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}],
        key=lambda p: p.name
    )
    all_paths = all_paths[::stride]

    for idx, p in enumerate(all_paths):
        img = cv2.imread(str(p))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        t   = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        yield idx, t, str(p)


def get_optimal_stride(actual_frame_count: int) -> int:
    """Determine the stride based on the ACTUAL number of annotated frames."""
    if actual_frame_count <= 1000:  return 1
    if actual_frame_count <= 2000:  return 2
    return 3


# ── Count actual frames and compute optimal stride ────────────────────────
print("\n[STEP A] Counting actual annotated frames per video ...")
print(f"  {'Video':<30} {'Actual':>8} {'Split sum':>10} {'Stride':>7} {'Est scan':>9}")
print("─" * 65)

full_video_meta: Dict[str, Dict] = {}

for vname in VIDEO_NAMES:
    actual = count_actual_frames(DATASET_ROOT, vname)
    # Verification: total split from video_test_files (if Cell 2 has been executed)
    # train + val + test should equal the actual count
    stride     = get_optimal_stride(actual)
    est_frames = (actual + stride - 1) // stride

    full_video_meta[vname] = {
        "actual_frames": actual,
        "stride"        : stride,
        "est_frames"    : est_frames,
    }
    print(f"  {vname:<30} {actual:>8} {'(see split)':>10} {stride:>7} {est_frames:>9}")

print()
total_actual = sum(v["actual_frames"] for v in full_video_meta.values())
total_scan   = sum(v["est_frames"]    for v in full_video_meta.values())
print(f"  {'TOTAL':<30} {total_actual:>8} {'':>10} {'':>7} {total_scan:>9}")
print(f"""
[OK] Lazy iterator ready — zero frames loaded into memory.
     Actual frame count = number of annotated frames (train + val + test).
     Frames are read, evaluated, and freed on-the-fly during the evaluation loop (Cell C).
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL B — VIDEO-LEVEL & TRACKING-AWARE METRICS
# 1. VIDEO-LEVEL DETECTION RATE (VDR)
#    Binary metric: Is the video detected?
#    VDR_clean  = 1 if ≥1 frame in the video has box_clean > 0
#    VDR_adv    = 1 if ≥1 frame in the video has box_adv   > 0
#    → If VDR_adv=1: The UAP fails at the video level (tracking may register it)
#
# 2. TEMPORAL EVASION STREAK (TES)
#    Longest CONSECUTIVE sequence of frames where na=0 (complete evasion)
#    Unit: number of frames & seconds (fps=25)
#    Significance: How long is the detector continuously "blind"?
#    → TES_max_sec > 2s = sufficient for the object to pass without tracking initialization
#
# 3. TRACKING-AWARE SUPPRESSION (TAS)
#    Simulates a simple tracker (IoU-based association):
#    An object is considered "tracked" if detected in ≥K_CONFIRM consecutive frames
#    (K_CONFIRM=3, standard for SORT/ByteTrack)
#    TAS = number of objects truly bypassed by the tracker / total objects
#    → High TAS = UAP is effective even when considering tracking
#
# 4. FRAME-SKIP RATIO (FSR)
#    In a sliding window of W frames:
#    FSR = |{windows where ALL frames evade}| / |total windows|
#    Measures: Does the UAP create a sufficiently long "detection gap" for the object to pass through?
#
# 5. FIRST-DETECT DELAY (FDD)
#    Number of frames to wait from the start of the video until the detector first
#    captures the object (clean vs. adv). Measures the ability to delay detection initialization.
# ═══════════════════════════════════════════════════════════════════════════

# Hyperparameters for tracking simulation
K_CONFIRM    = 3     # Number of consecutive frames for the tracker to confirm an object (SORT default)
FPS_ASSUMED  = 25    # Assumed fps to convert frames to seconds
WINDOW_SIZE  = 10    # W for Frame-Skip Ratio (10 frames ≈ 0.4s @ 25fps)


def compute_detection_sequence(
        frames      : List[torch.Tensor],
        delta       : torch.Tensor,
        yolo_wrapper: EvalYOLO,
        target_cls  : set,
        conf_thr    : float = CONF_THRESHOLD,
) -> Tuple[List[int], List[int]]:
    """
    Run the detector over the entire sequence of frames.
    Returns (nc_seq, na_seq): lists of clean and adversarial bounding box counts per frame.
    """
    nc_seq, na_seq = [], []
    for x_clean in tqdm(frames, desc="  Scanning sequence", leave=False):
        x_adv = torch.clamp(x_clean + delta, 0.0, 1.0)
        nc    = yolo_wrapper.count_boxes(x_clean, target_cls, conf_thr)
        na    = yolo_wrapper.count_boxes(x_adv,   target_cls, conf_thr)
        nc_seq.append(nc)
        na_seq.append(na)
    return nc_seq, na_seq


def compute_video_level_metrics(
        nc_seq     : List[int],    # Clean box count per frame
        na_seq     : List[int],    # Adversarial box count per frame
        k_confirm  : int   = K_CONFIRM,
        fps        : float = FPS_ASSUMED,
        window_size: int   = WINDOW_SIZE,
) -> Dict:
    """
    Compute all video-level metrics from the two sequences of nc and na.
    """
    T = len(nc_seq)
    assert T == len(na_seq), "nc_seq and na_seq must have the same length"

    # ── 1. VIDEO-LEVEL DETECTION RATE ─────────────────────────────────────
    vdr_clean = int(any(n > 0 for n in nc_seq))   # 1 if the video is detected (clean)
    vdr_adv   = int(any(n > 0 for n in na_seq))   # 1 if the video is still detected (adversarial)
    # vdr_adv=0 → The UAP is completely successful at the video level
    # vdr_adv=1 → The tracking system still has a chance to register it

    # Frame-level detection rate (for comparison)
    n_frames_w_obj = sum(1 for n in nc_seq if n > 0)
    fdr_clean = n_frames_w_obj / max(T, 1)   # Frame detection rate (clean)
    fdr_adv   = sum(1 for n in na_seq if n > 0) / max(T, 1)  # (adversarial)

    # ── 2. TEMPORAL EVASION STREAK ────────────────────────────────────────
    # Computed only on frames with actual objects (nc>0) → evasion is meaningful
    max_streak     = 0
    cur_streak     = 0
    streak_history = []   # List of lengths of all streaks

    for nc, na in zip(nc_seq, na_seq):
        if nc > 0:        # Frame contains an actual object
            if na == 0:   # And the adversarial example evades completely
                cur_streak += 1
            else:         # Detected again → streak ends
                if cur_streak > 0:
                    streak_history.append(cur_streak)
                max_streak = max(max_streak, cur_streak)
                cur_streak = 0
        # nc=0: Frame without an object → not counted in the streak
    if cur_streak > 0:
        streak_history.append(cur_streak)
        max_streak = max(max_streak, cur_streak)

    tes_max_frames = max_streak
    tes_max_sec    = max_streak / fps
    tes_mean_sec   = (np.mean(streak_history) / fps) if streak_history else 0.0
    tes_n_streaks  = len(streak_history)

    # ── 3. TRACKING-AWARE SUPPRESSION (simplified SORT simulation) ────────
    # Logic: The tracker requires K_CONFIRM consecutive frames with detections to "confirm" an object
    # If the adversarial attack causes misses for ≥ K_CONFIRM consecutive frames → the tracker fails to initialize or loses the track
    # 
    # We use a simplified simulation:
    #   - Track state: TENTATIVE (unconfirmed) or CONFIRMED
    #   - Transitions: TENTATIVE + detect×K → CONFIRMED
    #                  CONFIRMED + miss×1   → lost (reset)
    #   - TAS = 1 - (n_confirmed_adv / n_confirmed_clean)

    def simulate_tracker(det_seq: List[int], k: int) -> int:
        """Return the number of times the tracker CONFIRMS a track."""
        confirmed_count = 0
        tentative       = 0
        is_confirmed    = False
        for n in det_seq:
            if n > 0:
                if is_confirmed:
                    pass  # Track continues
                else:
                    tentative += 1
                    if tentative >= k:
                        is_confirmed    = True
                        confirmed_count += 1
            else:
                # Miss → reset
                tentative    = 0
                is_confirmed = False
        return confirmed_count

    tracker_confirms_clean = simulate_tracker(nc_seq, k_confirm)
    tracker_confirms_adv   = simulate_tracker(na_seq, k_confirm)

    # TAS = ratio of tracks prevented from being confirmed
    tas = 1.0 - (tracker_confirms_adv / max(tracker_confirms_clean, 1))
    tas = max(0.0, min(1.0, tas))  # Clamp to [0, 1]

    # ── 4. FRAME-SKIP RATIO (sliding window) ─────────────────────────────
    # Count the number of sliding windows of W frames where NA=0 in ALL frames of the window
    # (and at least 1 frame has NC>0 → the window contains an actual object)
    n_windows_w_obj  = 0
    n_windows_evade  = 0
    for start in range(T - window_size + 1):
        win_nc = nc_seq[start : start + window_size]
        win_na = na_seq[start : start + window_size]
        if any(n > 0 for n in win_nc):     # Window contains an actual object
            n_windows_w_obj += 1
            if all(n == 0 for n in win_na):  # All frames in the window evade
                n_windows_evade += 1

    fsr = n_windows_evade / max(n_windows_w_obj, 1)   # ↑ Better for the attacker

    # ── 5. FIRST-DETECT DELAY ────────────────────────────────────────────
    def first_detect_frame(seq: List[int]) -> Optional[int]:
        for i, n in enumerate(seq):
            if n > 0: return i
        return None

    fdd_clean = first_detect_frame(nc_seq)
    fdd_adv   = first_detect_frame(na_seq)
    fdd_delay = None
    if fdd_clean is not None and fdd_adv is not None:
        fdd_delay = fdd_adv - fdd_clean  # >0: Adversarial attack delays detection
    elif fdd_clean is not None and fdd_adv is None:
        fdd_delay = T - fdd_clean   # Never detected → maximum delay

    return {
        # -- Video-Level Detection Rate --
        "VDR_clean"          : vdr_clean,        # 1 = detected in video
        "VDR_adv"            : vdr_adv,           # 1 = still detected (attack failed at video-level)
        "FDR_clean"          : round(fdr_clean, 4),   # Frame-level detection rate (clean)
        "FDR_adv"            : round(fdr_adv, 4),     # Frame-level detection rate (adversarial)
        "frame_suppress_pct" : round((fdr_clean - fdr_adv) / max(fdr_clean, 1e-8) * 100, 2),
        # -- Temporal Evasion Streak --
        "TES_max_frames"     : tes_max_frames,
        "TES_max_sec"        : round(tes_max_sec, 2),
        "TES_mean_sec"       : round(tes_mean_sec, 2),
        "TES_n_streaks"      : tes_n_streaks,
        # -- Tracking-Aware Suppression --
        "TAS"                : round(tas, 4),          # ↑ Better (1 = tracker never confirms)
        "tracker_clean"      : tracker_confirms_clean,
        "tracker_adv"        : tracker_confirms_adv,
        # -- Frame-Skip Ratio --
        "FSR"                : round(fsr, 4),          # ↑ Better
        "FSR_window"         : window_size,
        # -- First-Detect Delay --
        "FDD_clean_frame"    : fdd_clean,
        "FDD_adv_frame"      : fdd_adv,
        "FDD_delay_frames"   : fdd_delay,
        "FDD_delay_sec"      : round(fdd_delay / fps, 2) if fdd_delay is not None else None,
        # -- Raw sequences (for plotting) --
        "_nc_seq"            : nc_seq,
        "_na_seq"            : na_seq,
    }


print("[OK] compute_video_level_metrics defined.")
print("""
  Video-Level Metrics:
  VDR_adv   : Video-Level Detection Rate (adv)  — 0 = attack won at video-level ↓
  FDR       : Frame Detection Rate               — frames detected / total frames
  TES_max   : Maximum Temporal Evasion Streak    — longest consecutive evasion sequence ↑
  TAS       : Tracking-Aware Suppression         — % of tracks prevented from confirmation ↑
  FSR       : Frame-Skip Ratio (window={:d}f)    — % of windows with complete evasion ↑
  FDD       : First-Detect Delay                 — number of frames/seconds of detection delay ↑
""".format(WINDOW_SIZE))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL PRE-C — Free VRAM before running the full-video evaluation
# Execute this cell BEFORE Cell C
# ═══════════════════════════════════════════════════════════════════════════
import gc

# 1. Clear all large tensors remaining from previous cells
vars_to_clear = ["test_frames", "frame_paths", "all_results_tensors"]
for vname in vars_to_clear:
    if vname in globals():
        del globals()[vname]

# 2. Remove per_frame raw data from all_results (retain only aggregated metrics)
if "all_results" in globals():
    for r in all_results:
        r.pop("per_frame",       None)
        r.pop("conf_clean_list", None)
        r.pop("conf_adv_list",   None)
    print(f"[OK] Cleared per_frame data from {len(all_results)} results")

# 3. Remove delta tensors if they are still lingering in the global namespace
for vname in list(globals().keys()):
    obj = globals()[vname]
    if isinstance(obj, torch.Tensor) and obj.is_cuda:
        del globals()[vname]
        print(f"  [DEL] Freed CUDA tensor: {vname}")

# 4. Force garbage collection and flush CUDA cache
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# 5. Reload the YOLO model with memory-efficient settings
# Delete the old model first
if "eval_yolo" in globals():
    del eval_yolo
gc.collect()
torch.cuda.empty_cache()

# Reinitialize — the model is configured for memory efficiency
class EvalYOLO_Lite:
    """
    Lightweight version of EvalYOLO: performs inference directly on CPU tensors,
    without keeping any frames in memory.
    """
    def __init__(self, model_name=YOLO_MODEL, conf=CONF_THRESHOLD,
                 iou=IOU_THRESHOLD):
        self.conf = conf
        self.iou  = iou
        self._yolo = YOLO(model_name)
        # Avoid calling .to(device) for the model — let Ultralytics manage device placement automatically
        print(f"  [EvalYOLO_Lite] Loaded {model_name}")

    @torch.no_grad()
    def count_boxes(self, x_cpu: torch.Tensor, target_classes: set) -> int:
        """
        Args:
            x_cpu: (C, H, W) float tensor in [0, 1] located on the CPU.
        """
        rgb = (x_cpu.numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        res = self._yolo(
            bgr, verbose=False, conf=self.conf, iou=self.iou,
            classes=sorted(target_classes), imgsz=IMG_SIZE,
        )
        n = 0
        if res and res[0].boxes is not None:
            n = len(res[0].boxes)
        # Delete the result immediately to free memory
        del res, bgr, rgb
        return n

eval_yolo = EvalYOLO_Lite()

gc.collect()
torch.cuda.empty_cache()

# 6. Report VRAM usage
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
free      = total - allocated
print(f"\n[VRAM] Allocated : {allocated:.2f} GB")
print(f"[VRAM] Reserved  : {reserved:.2f} GB")
print(f"[VRAM] Free (est): {free:.2f} GB")
print(f"[VRAM] Total     : {total:.2f} GB")
print("\n[OK] Ready — proceed to execute Cell C.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL C 
# ═══════════════════════════════════════════════════════════════════════════
import gc

all_video_results: List[Dict] = []

print("\n" + "═"*70)
print("  [FULL-VIDEO EVAL] Streaming — Tracking-Aware Assessment")
print("═"*70)

for (video_name, class_mode), pert_path in sorted(uap_map.items()):
    print(f"\n{'─'*70}")
    print(f"  {video_name}  |  class={class_mode}")

    # ── Load the perturbation tensor (delta) to CPU ──────────────────
    payload   = torch.load(str(pert_path), map_location="cpu",
                           weights_only=False)
    delta_cpu = payload["delta"].cpu()   # Retain on CPU
    meta      = {
        "video_name" : payload.get("video_name", "?"),
        "class_mode" : payload.get("class_mode", "?"),
        "metrics"    : payload.get("metrics", {}),
        "hyperparams": payload.get("hyperparams", {}),
        "timestamp"  : payload.get("timestamp", "?"),
    }
    del payload
    gc.collect()

    target_cls = CLASS_MODE_TO_YOLO[class_mode]
    stride     = full_video_meta[video_name]["stride"]
    est_frames = full_video_meta[video_name]["est_frames"]

    # Compute stealthiness metrics from the CPU tensor
    map_val      = delta_cpu.abs().mean().item()
    nuc_norm     = sum(torch.linalg.svdvals(delta_cpu[c]).sum().item()
                       for c in range(3))
    invisible_pct= (delta_cpu.abs() < 8/255).float().mean().item() * 100.0

    # ── Accumulators ──────────────────────────────────────────────────
    nc_seq: List[int] = []
    na_seq: List[int] = []
    sample_frame_cpu  = None
    sample_target_idx = est_frames // 2

    torch.cuda.empty_cache()

    # ── STREAMING LOOP ─────────────────────────────────────────────────
    for frame_idx, x_cpu, fpath in tqdm(
            iter_video_frames(DATASET_ROOT, video_name,
                              img_size=IMG_SIZE, stride=stride),
            total=est_frames,
            desc=f"  {video_name[:20]}|{class_mode}",
    ):
        # Apply the perturbation on CPU → perform inference directly on CPU to minimize VRAM usage
        x_adv_cpu = torch.clamp(x_cpu + delta_cpu, 0.0, 1.0)

        nc = eval_yolo.count_boxes(x_cpu,     target_cls)
        na = eval_yolo.count_boxes(x_adv_cpu, target_cls)
        nc_seq.append(nc)
        na_seq.append(na)

        if frame_idx == sample_target_idx and sample_frame_cpu is None:
            sample_frame_cpu = x_cpu.clone()

        # Free memory immediately — ensure no tensors are retained on the GPU
        del x_adv_cpu
        if frame_idx % 200 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    torch.cuda.empty_cache()
    print(f"  [OK] Scanned {len(nc_seq)} frames")

    # ── Compute video-level metrics ───────────────────────────────────
    vm = compute_video_level_metrics(nc_seq, na_seq)

    frame_result = next(
        (r for r in all_results
         if r["video_name"]==video_name and r["class_mode"]==class_mode),
        {}
    )
    combined = {
        "video_name"       : video_name,
        "class_mode"       : class_mode,
        "n_frames"         : len(nc_seq),
        "MAP"              : round(map_val, 6),
        "nuc_norm"         : round(nuc_norm, 4),
        "invisible_pct"    : round(invisible_pct, 2),
        **vm,
        "advBR_test"       : frame_result.get("advBR"),
        "attack_rate_test" : frame_result.get("attack_rate"),
        "vanish_rate_test" : frame_result.get("vanish_rate"),
    }
    all_video_results.append(combined)

    # ── Display evaluation results ────────────────────────────────────
    verdict = ("✅ ATTACK WINS (tracker cannot confirm)"
               if vm["VDR_adv"] == 0
               else f"⚠ PARTIAL — still detected in {vm['FDR_adv']*100:.1f}% frames")
    print(f"""
  VERDICT: {verdict}
  ┌────────────────────────────────────────────────────────────
  │ Frames scanned : {len(nc_seq)} (stride={stride})
  │ VDR_clean/adv  : {vm['VDR_clean']} / {vm['VDR_adv']}
  │ FDR clean/adv  : {vm['FDR_clean']*100:.1f}% / {vm['FDR_adv']*100:.1f}%
  │ Suppress       : {vm['frame_suppress_pct']:.1f}%
  │ TES max        : {vm['TES_max_frames']}f = {vm['TES_max_sec']:.2f}s
  │ TAS            : {vm['TAS']*100:.1f}%
  │ FSR            : {vm['FSR']*100:.1f}% (W={WINDOW_SIZE}f)
  │ FDD            : {vm['FDD_delay_sec']}s
  └────────────────────────────────────────────────────────────""")

    # ── Generate temporal visualization ───────────────────────────────
    pair_output = OUTPUT_DIR / f"{video_name}__{class_mode}"
    pair_output.mkdir(exist_ok=True)

    fig, axes = plt.subplots(2, 1, figsize=(18, 8), facecolor="white",
                              gridspec_kw={"height_ratios": [3, 1.5]})
    fig.suptitle(
        f"Full-Video Temporal Analysis  |  {video_name}  |  class={class_mode}",
        fontsize=12, fontweight="bold"
    )
    xs = list(range(len(nc_seq)))

    ax = axes[0]
    ax.fill_between(xs, nc_seq, color="#2ecc71", alpha=0.35, label="Clean")
    ax.fill_between(xs, na_seq, color="#e74c3c", alpha=0.55, label="Adversarial")
    ax.plot(xs, nc_seq, "#2ecc71", lw=0.8, alpha=0.7)
    ax.plot(xs, na_seq, "#e74c3c", lw=0.8, alpha=0.9)
    # The orange highlight for evasion streaks has been omitted
    ax.set_ylabel("# Detected Boxes", fontsize=9)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_title("① Detection Count", fontsize=10)

    ax = axes[1]
    evade_bin  = [1 if (nc>0 and na==0) else 0
                  for nc, na in zip(nc_seq, na_seq)]
    detect_adv = [1 if na > 0 else 0 for na in na_seq]
    ax.bar(xs, evade_bin, color="#e74c3c", alpha=0.7, width=1.0,
           label="Evade (nc>0, na=0)")
    ax.bar(xs, [-d*0.4 for d in detect_adv], color="#2ecc71",
           alpha=0.6, width=1.0, label="Adv detected (↓)")
    ax.set_ylim(-0.5, 1.3)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["0", "Evade"])
    ax.set_title("② Frame-by-Frame Evasion", fontsize=10)
    ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()
    out_path = pair_output / "full_video_timeline.png"
    plt.savefig(str(out_path), dpi=130, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [VIZ] → {out_path.name}")
    
    # Clean up memory after processing each pair
    del delta_cpu, sample_frame_cpu, nc_seq, na_seq
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n[FULL-VIDEO EVAL] Completed evaluation for {len(all_video_results)} pairs.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL D — FINAL SUMMARY: Frame-Level vs Video-Level Comparison
# + Automatic analysis of the results' significance for surveillance tasks
# ═══════════════════════════════════════════════════════════════════════════

if not all_video_results:
    print("[WARN] No video-level results available. Please execute Cell C first.")
else:
    print("\n" + "═"*100)
    print(f"{'FINAL REPORT: UAP EFFECTIVENESS — FRAME-LEVEL vs VIDEO-LEVEL':^100}")
    print("═"*100)
    print(f"  {'Pair':<35} {'FDR_c':>6} {'FDR_a':>6} {'Supp%':>6} "
          f"{'VDR_a':>5} {'TES_s':>6} {'TAS%':>6} {'FSR%':>6} {'FDD_s':>6}")
    print("─"*100)

    for vm in sorted(all_video_results,
                     key=lambda r: (r["video_name"], r["class_mode"])):
        pair = f"{vm['video_name'][:26]} | {vm['class_mode']}"
        fdd_s = vm["FDD_delay_sec"] if vm["FDD_delay_sec"] is not None else "-"
        print(f"  {pair:<35} "
              f"{vm['FDR_clean']*100:>5.1f}% "
              f"{vm['FDR_adv']*100:>5.1f}% "
              f"{vm['frame_suppress_pct']:>5.1f}% "
              f"{'YES' if vm['VDR_adv']==0 else 'no':>5} "
              f"{vm['TES_max_sec']:>6.2f}s "
              f"{vm['TAS']*100:>5.1f}% "
              f"{vm['FSR']*100:>5.1f}% "
              f"{str(fdd_s):>6}")

    print("─"*100)
    print("""
  LEGEND:
  FDR_c/a  : Frame Detection Rate (clean/adv) — % of frames detected
  Supp%    : % reduction in FDR due to UAP
  VDR_a    : Video-Level Detection (YES = attack completely failed at video-level)
  TES_s    : Max Evasion Streak (seconds) — longest continuous "blind" sequence
  TAS%     : Tracking-Aware Suppression — % of tracks prevented from confirmation (K=3 frames)
  FSR%     : % of sliding windows (10 frames) where adversarial examples completely evade
  FDD_s    : Delay before the first detection (seconds)
""")

    # ── Automatic analysis of the surveillance task ───────────────────────
    print("═"*100)
    print("  SURVEILLANCE CONTEXT ANALYSIS (Automatic)")
    print("═"*100)

    for vm in all_video_results:
        pair = f"{vm['video_name']} | {vm['class_mode']}"
        remarks = []

        # VDR check
        if vm["VDR_adv"] == 1:
            remarks.append(
                f"⚠  [VDR] The object is STILL detected in {vm['FDR_adv']*100:.1f}% "
                f"of frames in the video. Despite a frame-level suppression of {vm['frame_suppress_pct']:.0f}%, "
                f"the tracking system MAY still register this object."
            )
        else:
            remarks.append(
                f"✅ [VDR] The object is NOT detected in ANY frame. "
                f"The attack is completely successful at the video level."
            )

        # TES check (2s threshold ≈ typical object crossing an intersection)
        if vm["TES_max_sec"] >= 2.0:
            remarks.append(
                f"✅ [TES] Max streak {vm['TES_max_sec']:.1f}s ≥ 2s → "
                f"The object can pass through the observation area without track initialization."
            )
        else:
            remarks.append(
                f"⚠  [TES] Max streak is only {vm['TES_max_sec']:.1f}s < 2s → "
                f"The tracker may still initiate tracking before the object leaves the frame."
            )

        # TAS check
        if vm["TAS"] >= 0.8:
            remarks.append(
                f"✅ [TAS] {vm['TAS']*100:.0f}% of tracks prevented from confirmation "
                f"(K={K_CONFIRM}) → UAP is effective even against tracking."
            )
        elif vm["TAS"] >= 0.5:
            remarks.append(
                f"⚠  [TAS] Only {vm['TAS']*100:.0f}% of tracks prevented → "
                f"UAP is partially effective; some tracks are still confirmed."
            )
        else:
            remarks.append(
                f"❌ [TAS] Only {vm['TAS']*100:.0f}% of tracks prevented → "
                f"The tracker is barely affected. UAP is weak at the tracking level."
            )

        print(f"\n  [{pair}]")
        for r in remarks:
            print(f"    {r}")

    # ── Comprehensive comparison charts ────────────────────────────────────
    labels   = [f"{r['video_name'][:14]}\n{r['class_mode']}"
                for r in all_video_results]
    fdr_c    = [r["FDR_clean"]*100       for r in all_video_results]
    fdr_a    = [r["FDR_adv"]*100         for r in all_video_results]
    tes_vals = [r["TES_max_sec"]         for r in all_video_results]
    tas_vals = [r["TAS"]*100             for r in all_video_results]
    fsr_vals = [r["FSR"]*100             for r in all_video_results]

    x = np.arange(len(labels))
    w = 0.2

    fig, axes = plt.subplots(1, 3, figsize=(max(18, len(labels)*3), 6),
                              facecolor="white")
    fig.suptitle(
        "UAP Evaluation — Video-Level Metrics Summary\n"
        "Frame-level suppression vs. Tracking-aware effectiveness",
        fontsize=13, fontweight="bold"
    )

    # Sub-plot 1: FDR clean vs adv
    ax = axes[0]
    ax.bar(x - w/2, fdr_c, w*1.5, color="#2ecc71", alpha=0.85,
           label="FDR Clean (%)")
    ax.bar(x + w/2, fdr_a, w*1.5, color="#e74c3c", alpha=0.85,
           label="FDR Adv (%)")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("Frame Detection Rate (%)"); ax.set_ylim(0, 110)
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3, ls="--")
    ax.set_title("① Frame Detection Rate\n(lower adv = better)", fontsize=10)
    for xi, (c, a) in enumerate(zip(fdr_c, fdr_a)):
        ax.text(xi - w/2, c+1, f"{c:.0f}%", ha="center", fontsize=7, color="#27ae60")
        ax.text(xi + w/2, a+1, f"{a:.0f}%", ha="center", fontsize=7, color="#c0392b")

    # Sub-plot 2: TES & FSR
    ax = axes[1]
    ax.bar(x - w/2, tes_vals, w*1.5, color="#e67e22", alpha=0.85,
           label="TES max (sec)")
    ax2_twin = ax.twinx()
    ax2_twin.bar(x + w/2, fsr_vals, w*1.5, color="#9b59b6", alpha=0.6,
                 label="FSR (%)")
    ax.axhline(2.0, color="red", ls="--", lw=1.5, alpha=0.7,
               label="2s threshold")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("TES Max (seconds)", color="#e67e22")
    ax2_twin.set_ylabel("FSR (%)", color="#9b59b6")
    ax.set_title("② Temporal Evasion Streak & FSR\n(higher = longer blind window)",
                 fontsize=10)
    lines1, lab1 = ax.get_legend_handles_labels()
    lines2, lab2 = ax2_twin.get_legend_handles_labels()
    ax.legend(lines1+lines2, lab1+lab2, fontsize=8, loc="upper right")

    # Sub-plot 3: TAS
    ax = axes[2]
    bar_colors = ["#27ae60" if v >= 80 else "#e67e22" if v >= 50 else "#e74c3c"
                  for v in tas_vals]
    bars = ax.bar(x, tas_vals, width=0.5, color=bar_colors, alpha=0.88)
    ax.axhline(80, color="#27ae60", ls="--", lw=1.5, alpha=0.7,
               label="80% threshold (strong)")
    ax.axhline(50, color="#e67e22", ls="--", lw=1.5, alpha=0.7,
               label="50% threshold (partial)")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 115); ax.set_ylabel("TAS (%)")
    ax.set_title(f"③ Tracking-Aware Suppression (K={K_CONFIRM})\n"
                 f"(higher = tracker blinded)", fontsize=10)
    ax.legend(fontsize=8)
    for bar, v in zip(bars, tas_vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+1,
                f"{v:.0f}%", ha="center", fontsize=8, fontweight="bold")

    plt.tight_layout()
    final_chart = OUTPUT_DIR / "video_level_summary.png"
    plt.savefig(str(final_chart), dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"\n  [VIZ] Video-level summary chart → {final_chart.name}")

    # ── Save full results to JSON ──────────────────────────────────────────
    import json as _json
    json_out = OUTPUT_DIR / "video_level_metrics.json"
    # Exclude keys starting with '_' (raw sequences — too large)
    clean_results = [
        {k: v for k, v in r.items() if not k.startswith("_")}
        for r in all_video_results
    ]
    json_out.write_text(_json.dumps(clean_results, indent=2, ensure_ascii=False))
    print(f"  [JSON] Video-level metrics → {json_out.name}")

    print("\n[DONE] Full-video evaluation complete.")
    print(f"  → {final_chart}")
    print(f"  → {json_out}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL ZIP — COMPRESS AND EXPORT RESULTS
# ═══════════════════════════════════════════════════════════════════════════
import zipfile
import os
from datetime import datetime

timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"uap_evaluation_results_{timestamp}.zip"
zip_path     = OUTPUT_DIR / zip_filename

INCLUDE_EXTENSIONS = {
    ".png", ".jpg", ".jpeg",
    ".xlsx",
    ".json",
    ".txt",
    ".csv",
}

EXCLUDE_DIRS = {
    "dataset_eval_split", 
}

print(f"\n[ZIP] Compressing results → {zip_filename}")
print(f"      Source: {OUTPUT_DIR}")
print(f"      Excluded directories: {EXCLUDE_DIRS}")

total_files = 0
total_bytes = 0
skipped     = 0

with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED,
                     compresslevel=6) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        # Exclude unnecessary directories from the directory traversal
        dirs[:] = [
            d for d in dirs
            if d not in EXCLUDE_DIRS and d != "__pycache__"
        ]

        for fname in sorted(files):
            fpath = Path(root) / fname

            # Skip the zip file currently being written
            if fpath == zip_path:
                continue

            ext = fpath.suffix.lower()
            if ext not in INCLUDE_EXTENSIONS:
                skipped += 1
                continue

            arcname = fpath.relative_to(OUTPUT_DIR)
            zf.write(str(fpath), arcname=str(arcname))
            fsize = fpath.stat().st_size
            total_files += 1
            total_bytes += fsize
            print(f"  + {arcname}  ({fsize/1024:.1f} KB)")

zip_size = zip_path.stat().st_size
ratio    = (1 - zip_size / max(total_bytes, 1)) * 100

print(f"""
[ZIP] ✅ Completed!
      File     : {zip_path}
      Files    : {total_files}
      Raw size : {total_bytes/1024/1024:.2f} MB
      Zip size : {zip_size/1024/1024:.2f} MB  (compression: {ratio:.1f}%)
      Skipped  : {skipped} files (unsupported extension or in excluded dirs)
""")

try:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))
except Exception:
    print(f"  → Download at: {zip_path}")